[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session4/BMEN35_Ex4_01_Self_attention_NumPy.ipynb)

# Self-attention, one token at a time

Use a CPU runtime in Colab and run the cells in order. We will update the numerical representation of just **one** word, **chase**, in the sentence **cats chase mice**. We first mix three vectors using **attention weights** that we choose ourselves. Then we calculate those weights the way a Transformer does. You can follow the main example with basic arithmetic. The final two sections are optional.

## Where does self-attention fit in a Transformer?

A Transformer first converts the text to tokens and looks up a vector for each token ID. This initial vector is called a **token embedding**. The embedding table is a matrix with one row per vocabulary entry. Its numbers are learned during training. The model also uses information about each token's position in the sequence.

Transformer layers repeatedly update the representation at every position. Self-attention is the part that lets a position take in information from other positions in the same sequence. Consider **chase** in **cats chase mice**. The other words supply context about who does the chasing and what is being chased. Attention gives the model a way to combine information from those positions in a new vector for **chase**.

The original lookup vector is the **token embedding**. The vectors inside later layers are often called **hidden states** or **contextual representations** because they already include information from the surrounding text. Some texts also call those vectors *contextual embeddings*. In this notebook we use *token embedding* for the lookup vector and *representation* for a vector inside a layer.

The model learns how to produce the vectors used in attention during training. The **attention weights** for a particular sentence are calculated from those vectors each time the model processes the sentence. Later layers can use the resulting representations to classify the text or score possible next tokens. In this example all three positions are initially visible. The optional section at the end shows how generation hides future positions.

## 0. From text to tokens, IDs and embeddings

A **token** is a piece of text that a tokenizer recognises. It can be a whole word, part of a word or punctuation. In this toy vocabulary each of our three words is a single token.

A **token ID** is the integer used to look up a token in the vocabulary. We use `cats → 4`, `chase → 9` and `mice → 2`. These IDs are arbitrary. ID 9 tells us we would look up row 9 in the embedding table. A token's **position** says where it occurs in the sequence. For example, **chase** is at Python index 1 here, even though its token ID is 9.

A **token embedding** is the learned vector in the row for that ID. If **chase** occurs twice, both occurrences start from the same lookup vector. They can have different representations after the model uses their positions and surrounding context. Attention works with those numerical vectors. The example below only shows the IDs. We supply the vectors by hand in section 1.

In [ ]:
import numpy as np                            # We will use NumPy arrays for the vectors in the next sections.

sentence = 'cats chase mice'                   # The original text we want to represent.
tokens = sentence.split()                     # For this toy example only, split at spaces to get three tokens.
vocabulary = {'cats': 4, 'chase': 9, 'mice': 2}  # Give each token a made-up vocabulary ID.
token_ids = []                                # Store IDs in the same order as their tokens.

for position, token in enumerate(tokens):     # Enumerate gives the position and token, starting at index zero.
    token_id = vocabulary[token]              # Look up the ID for this token.
    token_ids.append(token_id)                # Store the integer in sequence order.
    print(f'Position {position} | token {token!r} | ID {token_id}')  # Display the three different ideas together.

print('ID sequence:', token_ids)               # These IDs are how a tokenizer can encode the text.

## 1. Start with known attention weights

A **value vector** is the information that one position can contribute to the result of attention. In a trained Transformer, each position has a current representation. A learned matrix, usually written `W_V`, transforms that representation into a value vector. The operation is called a **value projection**. In the first layer the representation starts from the token embedding and position information. In later layers the representation has already been updated by earlier layers.

For this walkthrough the three value vectors are made up. For example, pretend the representation for **mice** is `[2, 2]` and a made-up projection doubles both components. That would give the value `[4, 4]`. A trained model learns the coefficients in its projection matrix and can mix different components.

For now, we also choose the **attention weights** ourselves. Give **cats** weight 0.25, **chase** weight 0.25 and **mice** weight 0.50. Each weight says how strongly its position's value contributes **to the output for this one query**. They are all nonnegative and add up to one.

Multiply each value vector by its **attention weight**, then add the results component by component. The contribution from **cats** is `0.25 × [2, 0] = [0.5, 0]`. The three contributions add up to `[0.5, 0] + [0, 0.5] + [2, 2] = [2.5, 2.5]`. Before running the code, identify which token contributes most to the first component.

In [ ]:
values = np.array([                                        # Use invented value vectors in sentence order.
    [2., 0.],                                             # Row 0 is what cats can contribute.
    [0., 2.],                                             # Row 1 is what chase can contribute.
    [4., 4.],                                             # Row 2 is what mice can contribute.
])
example_attention_weights = np.array([0.25, 0.25, 0.50])   # Choose one attention weight per source position.

from_cats = example_attention_weights[0] * values[0]       # Scale both parts of the cats value by 0.25.
from_chase = example_attention_weights[1] * values[1]      # Scale both parts of the chase value by 0.25.
from_mice = example_attention_weights[2] * values[2]       # Scale both parts of the mice value by 0.50.
mixed_value = from_cats + from_chase + from_mice             # Add the weighted contributions component by component.

print('Contribution from cats: ', from_cats)                # Show [0.5, 0].
print('Contribution from chase:', from_chase)                # Show [0, 0.5].
print('Contribution from mice: ', from_mice)                 # Show [2, 2].
print('Weighted sum:', mixed_value)                         # Show [2.5, 2.5], the output for chase.

## 2. What exactly are query, key and value?

The next goal is to calculate **attention weights** instead of assigning them by hand. Every position can play two roles. It may be the position we want to **update**, and it may be a **source** of information for any position that can attend to it. We will update **chase** and look at three possible sources.

| Vector | Where it comes from | What we do with it |
| --- | --- | --- |
| **Query**, Q | From the current representation of the position we are updating. Here we use the query for **chase**. | Compare this one query with each source's key. It specifies what kind of information this position is looking for in this attention head. |
| **Key**, K | From the current representation at each possible source. There is a key for **cats**, one for **chase** and one for **mice**. | Compare it with the query using a dot product. A bigger match score means that source gets a larger attention weight after softmax. |
| **Value**, V | From the current representation at each possible source. Each source has a value as well as a key. | Multiply this vector by the source's **attention weight** and add it to the other weighted values. This vector is the information being mixed. |

If the source **mice** has key `[2, 2]` and value `[4, 4]`, we compare the query with `[2, 2]`, then multiply the resulting attention weight by `[4, 4]`. This is why changing a value changes the information passed on while leaving the weight calculation for that layer unchanged.

A Transformer has separate learned projection matrices for these three roles. For a representation `x` written as a row vector, we can write `query = x @ W_Q`, `key = x @ W_K` and `value = x @ W_V`. The three matrices can extract different information from the same input. It is **self**-attention because these inputs all come from positions in the same sequence. Our toy code uses one set of projections conceptually, called one attention head, and supplies the results of those projections directly.

In [ ]:
query_chase = np.array([1., 1.])              # Use a two-component query for the position we update.
keys = np.array([                             # List the keys in the same order as the values.
    [1., 0.],                                 # Key for cats.
    [0., 1.],                                 # Key for chase.
    [2., 2.],                                 # Key for mice.
])

print('Query for chase:', query_chase)         # This one query will be used in all three comparisons.
for index in range(len(tokens)):              # Visit the three positions using indices 0, 1 and 2.
    print(tokens[index], '| key', keys[index], '| value', values[index])  # Show each token with its two vectors.

## 3. Calculate a score for each possible source

A **dot product** compares two equal-length vectors. Multiply numbers in matching positions, then add the products. For **cats**, the calculation is `1 × 1 + 1 × 0 = 1`. For **chase**, the calculation is `1 × 0 + 1 × 1 = 1`. Calculate the dot product for **mice** before running the cell. There should be three scores, one per possible source. These are intermediate numbers, so they do not need to add up to 1.

Transformers use **scaled dot-product attention**. Divide the dot products by the square root of the number of components in a key. Here we divide by the square root of 2, about 1.41. Dividing by the same number preserves the order of the scores. The scale matters more for long vectors, where raw dot products can grow and make softmax concentrate too much weight on one position.

In [ ]:
dot_products = []                            # Collect one raw score per key, in token order.

for index in range(len(tokens)):              # Keep the query fixed and visit one key at a time.
    key = keys[index]                         # Select the two numbers in this position's key.
    score = np.dot(query_chase, key)           # Multiply matching components and sum the products.
    dot_products.append(score)                # Keep this score for the softmax calculation.
    print(f'chase compared with {tokens[index]}: {score:.0f}')  # Print a token name beside its own score.

key_size = keys.shape[1]                       # Number of columns in keys, so each key has two components.
scale = np.sqrt(key_size)                      # Square root of the key size, about 1.41 here.
scaled_scores = np.array(dot_products) / scale  # Convert to an array and divide every score by the same scale.
print('Scaled scores:', np.round(scaled_scores, 3))  # Display rounded values; keep full precision for the next cell.

## 4. Turn the scores into attention weights with softmax

We want a set of **attention weights** like the ones in section 1. They are nonnegative and sum to 1. **Softmax** provides these weights by exponentiating the scores and dividing each exponential by their sum. A larger score gives a larger weight. Equal scores give equal weights.

Follow the numbers for **chase**. The scaled scores are approximately `[0.707, 0.707, 2.828]`. In the code, we subtract the largest score to get `[-2.121, -2.121, 0]`. This step keeps the exponentials from getting too large. It multiplies all the exponentials by the same factor, which cancels when we divide by their sum. It therefore leaves the weights unchanged.

The exponentials are approximately `[0.120, 0.120, 1.000]`. Their sum is about 1.240. Dividing by that sum gives the approximate weights `[0.097, 0.097, 0.807]`. Their rounded versions may add up to slightly more or less than 1. Check the sum using the values before we round them.

These **attention weights** are over positions in the input. The text generation walkthrough uses softmax for a different purpose. There it turns scores for possible **next tokens** into probabilities.

In [ ]:
shifted_scores = scaled_scores - scaled_scores.max()  # Move the largest score to zero for numerical stability.
exponentials = np.exp(shifted_scores)                   # Apply the exponential to each of the three scores.
total = exponentials.sum()                            # This is the shared divisor, about 1.240 here.
weights_for_chase = exponentials / total              # Normalise the three weights so they add up to one.

for index in range(len(tokens)):                       # Print one token and its own weight at a time.
    print(f'{tokens[index]:>5s} | weight {weights_for_chase[index]:.3f}')  # Show three decimal places.
print('Sum before display rounding:', weights_for_chase.sum())          # Check the normalisation.

## 5. Use the weights to update the representation of chase

We repeat the operation from section 1 with the calculated **attention weights**. The output vector for **chase** comes from adding the three weighted value vectors. Its first component is about `0.097 × 2 + 0.097 × 0 + 0.807 × 4 = 3.42`. Compare every contribution with the printed weight for that position.

**Pause.** Suppose the value for **mice** changes to `[8, 8]` while the key stays at `[2, 2]`. Predict what happens to the weights and to the resulting vector. To check, change the value in section 1 and run all the code again through section 5. Restore `[4, 4]` afterwards. The short answers at the end explain why the weights and output behave differently.

In [ ]:
attention_output = np.zeros(2)               # Start with zero in each component of the output vector.

for index in range(len(tokens)):              # Visit the tokens in the same order as the weights and values.
    contribution = weights_for_chase[index] * values[index]  # Scale the value at this position by its weight.
    print(f'{tokens[index]:>5s} contributes {np.round(contribution, 3)}')  # Inspect this token's contribution.
    attention_output += contribution          # Add the two components to their matching output components.

print('Attention output for chase:', np.round(attention_output, 3))  # Read the new two-component vector.

## Optional 1. Update all three positions at the same time

Each position has its own query, so **cats** can have different weights from **chase**. A matrix lets us do the same operation for all three queries. In the score matrix, a row means **the position we are updating**, and a column means **the position supplying information**. A row gets its own softmax.

Both `queries` and `keys` have shape `(3, 2)`. Transposing the keys makes a `(2, 3)` matrix. The product `queries @ keys.T` therefore has shape `(3, 3)`. Compare the second row of the weight matrix with section 4.

In [ ]:
queries = np.array([                           # Each position now has its own query row.
    [1., 0.],                                  # Query for cats.
    [1., 1.],                                  # Query for chase from section 2.
    [0., 1.],                                  # Query for mice.
])
score_matrix = (queries @ keys.T) / np.sqrt(key_size)  # Compare all three queries with all three keys.
row_maxima = score_matrix.max(axis=1, keepdims=True)  # Find one maximum per row, retaining a column for it.
shifted_matrix = score_matrix - row_maxima            # Subtract each row's own maximum from that row.
exp_matrix = np.exp(shifted_matrix)                    # Exponentiate each score separately.
row_totals = exp_matrix.sum(axis=1, keepdims=True)      # Sum across columns so each query gets its own divisor.
weight_matrix = exp_matrix / row_totals                # Normalise each query row independently.

print('Rows and columns:', tokens)                     # Rows are queries and columns are source positions.
print(np.round(weight_matrix, 3))                      # The second row gives the weights for chase.
print('Output vectors in the same row order:')          # Announce the result for all three positions.
print(np.round(weight_matrix @ values, 3))              # Use each row of weights to mix the same three values.

## Optional 2. Stop a position from looking ahead

A model that learns to predict the next token should have access to the context that will be available when it generates text. If we are at **chase**, we can read **cats** and **chase** to predict the next token. We cannot read **mice** ahead of time. A **causal mask** blocks future positions.

For one row we can demonstrate it by replacing the score for **mice** with negative infinity before softmax. Its exponential becomes zero. The remaining weights still add up to one. Which weights do you expect for **chase** when its two available scores are equal?

In [ ]:
causal_scores_for_chase = scaled_scores.copy()   # Keep the original scores for the unmasked comparison.
causal_scores_for_chase[2] = -np.inf             # Block mice, which is to the right of chase.

shifted_causal = causal_scores_for_chase - causal_scores_for_chase.max()  # Shift the largest allowed score to zero.
causal_exp = np.exp(shifted_causal)               # The exponential of negative infinity is zero.
causal_weights = causal_exp / causal_exp.sum()    # Divide by the sum of the remaining exponentials.

print('Weights with all positions visible:', np.round(weights_for_chase, 3))  # Use the original weights.
print('Weights with future positions hidden:', np.round(causal_weights, 3))  # Confirm a zero weight for mice.
print('Output with the causal mask:', np.round(causal_weights @ values, 3))   # Mix the vectors using the new weights.

## Short answers to the pause questions

- With the example weights, **mice** contributes 2 to the first component, **cats** contributes 0.5 and **chase** contributes 0.
- The query for **chase** has a dot product of 4 with the key for **mice**. The dot products with the other two keys are both 1. That is why their weights match.
- Changing only the value for **mice** to `[8, 8]` leaves the weights unchanged. The output increases from about `[3.42, 3.42]` to `[6.65, 6.65]`, because the largest weight now multiplies larger numbers.
- With the causal mask, **cats** and **chase** each get weight 0.5 and **mice** gets weight zero. With the original values, the resulting vector is `[1, 1]`.

[Scaled dot-product attention in the original Transformer paper](https://proceedings.neurips.cc/paper/7181-attention-is-all-you-need.pdf).